# 003 OpenAI Structured Output

这是第三份 OpenAI 学习 Notebook。

学习目标：

1. 理解为什么业务接口不能只依赖自由文本
2. 学会用 Pydantic 定义结构化输出 schema
3. 学会用 OpenAI Python SDK 把模型输出直接解析成结构化对象
4. 理解私有兼容网关不支持 `parse(...)` 时的降级思路

这一阶段非常关键，因为真实业务更需要稳定字段，而不是一大段聊天文本。

以股票助手为例，你真正想拿到的是：

- `intent`
- `symbol`
- `time_range`
- `needs_analysis`

参考文档：

- OpenAI Structured Outputs：https://platform.openai.com/docs/guides/structured-outputs


## 先理解核心概念

普通聊天通常返回一段自然语言，例如：

```python
"AAPL 最近一周整体偏震荡，建议结合成交量继续观察。"
```

这对人类可读，但对程序不友好。

如果后端想继续调用工具，比如：

- 查实时行情
- 查 K 线数据
- 做技术分析

那更适合先让模型输出结构化结果，例如：

```python
{
    "intent": "stock_quote",
    "symbol": "AAPL",
    "time_range": "7d",
    "needs_analysis": True,
}
```

你可以把它理解成：

- 自由文本：像 `String`
- 结构化输出：像 Java 里的 DTO / VO

而 `Pydantic` 很像“带校验能力的 DTO”。


## 加载环境变量

这里和前两份 Notebook 保持一致：自动向上查找项目根目录 `.env`。


In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv


def load_project_env() -> Path | None:
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        env_path = path / ".env"
        if env_path.exists():
            load_dotenv(env_path, override=False)
            return env_path
    return None


env_path = load_project_env()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL") or None

print(f"Loaded .env: {env_path}" if env_path else "No .env found")
print("OPENAI_MODEL =", OPENAI_MODEL)
print("OPENAI_API_KEY loaded =", bool(OPENAI_API_KEY))
print("OPENAI_BASE_URL =", OPENAI_BASE_URL)


Loaded .env: /home/dev/bxc/fastapi-study/.env
OPENAI_MODEL = qwq
OPENAI_API_KEY loaded = True
OPENAI_BASE_URL = http://192.168.102.19:8082/v1


## 创建客户端

当前环境里 `openai` SDK 版本是 `2.36.0`，支持 `client.beta.chat.completions.parse(...)`。

如果你连接的是 OpenAI 官方服务，Structured Outputs 优先学习这条路径。

如果你连接的是私有兼容网关，是否支持 `parse(...)`、`json_schema`、`json_object`，取决于网关实现，不一定完全兼容。


In [2]:
from openai import OpenAI


if not OPENAI_API_KEY:
    raise ValueError("请先在项目根目录 .env 中配置 OPENAI_API_KEY")


client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
)


## 定义结构化输出 Schema

这一段最重要。

先不要急着调用模型，先把你想要的输出结构定义清楚。

这里我们定义一个“股票意图识别”对象：

- `intent`：意图类型
- `symbol`：股票代码
- `time_range`：时间范围
- `needs_analysis`：是否需要分析

这和 Java 里先定义 `StockIntentDTO` 再接收请求，很像。


In [3]:
from typing import Literal

from pydantic import BaseModel, Field


class StockIntent(BaseModel):
    intent: Literal["stock_quote", "stock_analysis", "general_chat", "unknown"] = Field(
        description="用户意图类型"
    )
    symbol: str | None = Field(default=None, description="股票代码，例如 AAPL、TSLA")
    time_range: str | None = Field(default=None, description="时间范围，例如 7d、30d、ytd")
    needs_analysis: bool = Field(description="是否需要进一步分析")


## 看看 Schema 的样子

`model_json_schema()` 可以把 Pydantic 模型转成 JSON Schema。

这也是为什么 Pydantic 很适合结构化输出：

- 对 Python 代码友好
- 对模型约束也友好


In [4]:
from pprint import pprint

pprint(StockIntent.model_json_schema())


{'properties': {'intent': {'description': '用户意图类型',
                           'enum': ['stock_quote',
                                    'stock_analysis',
                                    'general_chat',
                                    'unknown'],
                           'title': 'Intent',
                           'type': 'string'},
                'needs_analysis': {'description': '是否需要进一步分析',
                                   'title': 'Needs Analysis',
                                   'type': 'boolean'},
                'symbol': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
                           'default': None,
                           'description': '股票代码，例如 AAPL、TSLA',
                           'title': 'Symbol'},
                'time_range': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
                               'default': None,
                               'description': '时间范围，例如 7d、30d、ytd',
                               'title': 'Time

## 准备系统提示词

Structured Outputs 不是“只写 schema 就万事大吉”。

你仍然应该给模型一个清楚的业务角色和字段解释。

经验上，最稳的方式是：

- 用 system prompt 说明业务目标
- 用 Pydantic schema 约束输出结构
- 最后再对解析结果做程序侧校验


In [5]:
SYSTEM_PROMPT = """
你是一个股票助手的意图识别器。
用户会用中文描述自己的需求。

请根据用户输入识别以下字段：
1. intent:
   - stock_quote: 查询行情、走势、涨跌、价格
   - stock_analysis: 请求分析、解读、判断、建议
   - general_chat: 普通闲聊或泛化问题
   - unknown: 无法判断
2. symbol: 如果用户明确提到股票代码，则提取，例如 AAPL、TSLA、MSFT
3. time_range: 如果用户提到时间范围，则尽量标准化，例如 7d、30d、ytd
4. needs_analysis: 如果用户希望做解读、分析、建议，则为 true；否则为 false

回答时请严格遵守 schema。
""".strip()


## 官方首选写法：直接解析成 Pydantic 对象

OpenAI Python SDK 可以直接把返回结果解析成 `StockIntent`。

这一步的直觉可以理解成：

- Java：接口返回 JSON，再反序列化成 DTO
- Python + OpenAI SDK：模型返回内容后，SDK 直接帮你解析成 Pydantic 对象

重点看这行：

```python
response_format=StockIntent
```


In [13]:
def parse_stock_intent(message: str) -> StockIntent:
    completion = client.beta.chat.completions.parse(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": message},
        ],
        response_format=StockIntent,
    )

    choice = completion.choices[0]
    refusal = getattr(choice.message, "refusal", None)
    if refusal:
        raise ValueError(f"模型拒绝回答: {refusal}")

    parsed = choice.message.parsed
    if parsed is None:
        raise ValueError("模型没有返回可解析的结构化结果")

    return parsed


## 运行一个股票意图识别示例

这正对应学习计划里的目标：

```json
{
  "intent": "stock_quote",
  "symbol": "AAPL",
  "time_range": "7d",
  "needs_analysis": true
}
```

注意：`needs_analysis` 最终是 `true` 还是 `false`，取决于你给用户问题的措辞。


In [10]:
intent = parse_stock_intent("帮我看下 AAPL 最近一周的走势，并简单分析一下。")
intent


Raw model response:ParsedChatCompletionMessage[TypeVar](content='{\n  "intent": "stock_analysis",\n  "symbol": "AAPL",\n  "time_range": "7d",\n  "needs_analysis": true\n}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, parsed=StockIntent(intent='stock_analysis', symbol='AAPL', time_range='7d', needs_analysis=True), reasoning_content='Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "帮我看下 AAPL 最近一周的走势，并简单分析一下。" (Help me check the recent week\'s trend of AAPL, and briefly analyze it.)\n   - Key elements:\n     - Stock symbol: "AAPL"\n     - Time range: "最近一周" (recent week) -> standardizes to "7d" or "1w". I\'ll use "7d" as it\'s common in financial contexts, or "1w". The prompt says "例如 7d、30d、ytd", so "7d" is good.\n     - Intent: "走势" (trend) -> stock_quote. "分析一下" (analyze) -> stock_analysis. The combination usually points to stock_analysis because it explicitly asks for analysis/brief interpretation.\n   

StockIntent(intent='stock_analysis', symbol='AAPL', time_range='7d', needs_analysis=True)

## 转成普通字典

`StockIntent` 是 Pydantic 对象。

如果你后面要：

- 存数据库
- 返回 FastAPI 接口
- 继续调用工具函数

通常会先转成字典。


In [11]:
intent.model_dump()


{'intent': 'stock_analysis',
 'symbol': 'AAPL',
 'time_range': '7d',
 'needs_analysis': True}

## 再试两个不同场景

结构化输出的价值，在于你可以稳定地区分业务分支。

例如下面两条消息，虽然都是自然语言，但程序拿到的是统一结构。


In [14]:
examples = [
    "查一下 TSLA 今天的价格",
    "你觉得英伟达未来半年值得长期持有吗？",
]

for message in examples:
    parsed = parse_stock_intent(message)
    print("user:", message)
    print("parsed:", parsed.model_dump())
    print("-" * 60)


user: 查一下 TSLA 今天的价格
parsed: {'intent': 'stock_quote', 'symbol': 'TSLA', 'time_range': '1d', 'needs_analysis': False}
------------------------------------------------------------
user: 你觉得英伟达未来半年值得长期持有吗？
parsed: {'intent': 'stock_analysis', 'symbol': 'NVDA', 'time_range': '6m', 'needs_analysis': True}
------------------------------------------------------------


## 为什么这比自由文本更适合后续工具调用

有了结构化结果后，后端代码就可以走确定分支，例如：

```python
if parsed.intent == "stock_quote":
    get_stock_quote(parsed.symbol, parsed.time_range)
elif parsed.intent == "stock_analysis":
    run_stock_analysis(parsed.symbol, parsed.time_range)
```

如果模型只返回一大段自然语言，后端通常还要再写一层字符串解析，稳定性会明显更差。


## 拒答处理要单独考虑

官方文档提到，某些请求如果触发安全策略，模型可能不会返回符合 schema 的业务字段，而是返回拒答。

因此程序里不能只假设：

- 一定有 `parsed`
- 一定能成功反序列化

你已经在 `parse_stock_intent()` 里看到这段处理：

```python
refusal = getattr(choice.message, "refusal", None)
if refusal:
    raise ValueError(...)
```

真实项目里，这里通常会转成统一错误响应，而不是直接抛异常。


## 私有兼容网关的降级方案

如果你的兼容网关不支持：

- `client.beta.chat.completions.parse(...)`
- 或严格 schema 解析

那可以退一步：

1. 让模型返回 JSON 字符串
2. 再用 `Pydantic` 自己校验

这不如官方 Structured Outputs 稳定，但对很多早期项目已经够用。


In [ ]:
import json


def parse_stock_intent_fallback(message: str) -> StockIntent:
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
                + "
请只返回一个 JSON 对象，不要输出额外解释。",
            },
            {"role": "user", "content": message},
        ],
        response_format={"type": "json_object"},
    )

    raw_text = response.choices[0].message.content or "{}"
    data = json.loads(raw_text)
    return StockIntent.model_validate(data)


## 运行降级方案示例

如果你的网关支持 `json_object`，这段通常也能工作。

如果连 `json_object` 都不支持，那就只能继续退回“提示词要求输出 JSON + 本地校验”的方案，但稳定性会更差。


In [ ]:
fallback_intent = parse_stock_intent_fallback("帮我看下 MSFT 近一个月走势")
fallback_intent.model_dump()


## 对比两种方式

| 方式 | 优点 | 风险 |
|---|---|---|
| `parse(..., response_format=StockIntent)` | 最贴近官方 Structured Outputs，类型清晰，代码最干净 | 私有兼容网关可能不支持 |
| `json_object` + `Pydantic.model_validate(...)` | 更容易兼容部分网关 | 结构约束通常不如官方模式严格 |

如果你后面要做真实业务，建议优先选择能稳定保证 schema 的方式。


## 和 FastAPI 的关系

下一步接到 FastAPI 接口里时，通常会变成这样：

1. 接收用户输入 `message`
2. 调用 `parse_stock_intent()`
3. 根据 `intent` 选择业务分支
4. 再决定是否调用行情、分析、检索等工具

这就是从“聊天”走向“智能体”的第一步。


In [15]:
# 伪代码示例
message = "帮我看下 AAPL 最近一周的走势"
parsed = parse_stock_intent(message)

if parsed.intent == "stock_quote" and parsed.symbol:
    print(f"下一步可以调用 get_stock_quote(symbol={parsed.symbol}, time_range={parsed.time_range})")
elif parsed.intent == "stock_analysis" and parsed.symbol:
    print(f"下一步可以调用 run_stock_analysis(symbol={parsed.symbol}, time_range={parsed.time_range})")
else:
    print("继续澄清用户意图")


下一步可以调用 get_stock_quote(symbol=AAPL, time_range=7d)


## 当前阶段结论

你现在需要记住：

1. Structured Outputs 的目标是让模型返回“程序可消费的数据结构”
2. `Pydantic` 在这里就像 Java 里的 DTO + 校验器
3. 官方推荐路径是直接解析成 Pydantic 对象
4. 私有兼容网关如果能力不完整，就退回 JSON 字符串 + 本地校验
5. 这一层做好之后，后面才能更稳定地接函数调用和工具编排

下一份建议学习：

- Function Calling / Tool Calling
- 再把结构化输出接进 FastAPI 接口
